# Entrenamiento de toxicidad con TF-IDF y regresión logística

Reproduce la línea base del PR #5. Puedes leer las salidas guardadas o volver a ejecutar cada celda con el kernel de Python 3.12 de la `.venv`. El CSV original se carga desde `data/raw/` y se conserva localmente.

El objetivo es seguir el recorrido desde las particiones hasta el F1. El test de este ejercicio es **histórico** y sirve para reproducir un resultado conocido. Para comparar ajustes, cambia `reproducir_test_historico` a `False` antes de ejecutar.

## 1. Localizar los datos y elegir el modo

Abre el notebook desde JupyterLab iniciado en la raíz del repositorio. La celda también admite iniciar el kernel desde `notebooks/`. Cada ejecución crea un directorio propio bajo `data/local/`.

In [1]:
from datetime import datetime
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
if not (repo / "scripts/train_logistic_tfidf.py").is_file():
    raise FileNotFoundError("Abre el notebook desde la raíz del repositorio o desde notebooks/.")

dataset = repo / "data/raw/youtoxic_english_1000.csv"
manifest = repo / "data/splits/common_split.csv"
if not dataset.is_file():
    raise FileNotFoundError("Coloca el CSV autorizado en data/raw/youtoxic_english_1000.csv.")

reproducir_test_historico = True
marca = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
output_dir = repo / "data/local/experiments" / f"logistic-notebook-{marca}"

## 2. Inspeccionar la partición por vídeo

El manifiesto se une al CSV por `CommentId`. El módulo comprueba que los vídeos y los textos normalizados no se repitan entre particiones. La tabla muestra los tamaños de cada grupo. El resumen no muestra textos ni identificadores individuales.

In [2]:
sys.path.insert(0, str(repo))
from src.moderation.models.logistic_tfidf import load_partitions

particiones = load_partitions(dataset, manifest)
pd.DataFrame(
    [
        {"Partición": split, "Comentarios": len(filas)}
        for split, filas in particiones.items()
    ]
).set_index("Partición")

,Comentarios
Partición,
train,580
validation,219
test,185


## 3. Entrenar la línea base

El comando del repositorio se ejecuta desde esta celda con el Python del kernel. El pipeline aprende TF-IDF de fragmentos de 3 a 5 caracteres y ajusta la regresión logística con los comentarios de entrenamiento. En validación se elige el umbral más alto que alcanza el objetivo de recall del 80 %. Ese umbral se aplica a todas las particiones evaluadas.

Con el modo histórico activado, el script verifica la configuración versionada en Git antes de evaluar el test.

In [3]:
comando = [
    sys.executable, "scripts/train_logistic_tfidf.py",
    "--dataset", str(dataset),
    "--split", str(manifest),
    "--output-dir", str(output_dir),
]
if reproducir_test_historico:
    comando += ["--final-test", "--ensemble-config", "configs/ensemble.json"]

subprocess.run(comando, cwd=repo, check=True, stdout=subprocess.DEVNULL)
informe = json.loads((output_dir / "metrics.json").read_text(encoding="utf-8"))
print(f"Umbral fijado en validación: {informe['threshold']:.9f}")
print(f"Recall objetivo en validación: {informe['target_recall']:.0%}")

Umbral fijado en validación: 0.387493988
Recall objetivo en validación: 80%


## 4. Leer las métricas y calcular la brecha

La precisión indica qué proporción de las alertas corresponde a comentarios etiquetados como tóxicos. El recall indica qué proporción de los comentarios tóxicos se detectó. F1 resume ambas métricas. Para comparar entrenamiento y test, multiplicamos la diferencia absoluta de F1 por 100.

In [4]:
nombres = {"train": "Entrenamiento", "validation": "Validación", "test": "Test histórico"}
metricas = pd.DataFrame(
    {
        nombre: {
            "Precisión": f"{informe[split]['precision']:.2%}",
            "Recall": f"{informe[split]['recall']:.2%}",
            "F1": f"{informe[split]['f1']:.2%}",
        }
        for split, nombre in nombres.items()
        if split in informe
    }
).T
metricas.index.name = "Partición"
metricas

,Precisión,Recall,F1
Partición,,,
Entrenamiento,87.86%,100.00%,93.54%
Validación,66.89%,80.49%,73.06%
Test histórico,57.04%,87.50%,69.06%


### Comprobar el F1 de test con su matriz de confusión

Las filas son las etiquetas reales. Las columnas son las decisiones del modelo. Con los verdaderos positivos (VP), falsos positivos (FP) y falsos negativos (FN) puedes recalcular `F1 = 2 × VP / (2 × VP + FP + FN)`.

In [5]:
from IPython.display import display

if "test" in informe:
    matriz = informe["test"]["confusion_matrix"]
    display(pd.DataFrame(
        matriz,
        index=["Real: no tóxico", "Real: tóxico"],
        columns=["Marcado no tóxico", "Marcado tóxico"],
    ))
    brecha = informe["train_test_f1_gap_pp"]
    print(f"Brecha absoluta de F1 entre entrenamiento y test: {brecha:.2f} puntos porcentuales.")
    print("Cumple el objetivo de menos de 5 puntos." if brecha < 5 else
          "Supera el objetivo de menos de 5 puntos.")
else:
    print("La ejecución termina en validación. Activa el modo histórico para reproducir el test.")

,Marcado no tóxico,Marcado tóxico
Real: no tóxico,39,58
Real: tóxico,11,77


Brecha absoluta de F1 entre entrenamiento y test: 24.48 puntos porcentuales.
Supera el objetivo de menos de 5 puntos.


## 5. Conservar el experimento

Al guardar el notebook se conservan las celdas y sus salidas agregadas. La carpeta de la ejecución conserva `metrics.json`, el pipeline `logistic_tfidf.joblib`, sus metadatos y los CSV de predicciones. El CSV original, el modelo y las predicciones individuales quedan fuera de Git. Para una comparación de hiperparámetros, usa entrenamiento y validación y reserva vídeos independientes para la confirmación final.

[Informe de la línea base](../docs/model/logistic-tfidf.md#reproducción-del-test-histórico)